# Phase 4 — Agent A3 : Sentiment Analyser (PRD v3.0 — ReAct + Tools)
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre l'agent A3 (`agents/a3_sentiment.py`) en version **PRD v3.0** :
- **ReAct** (Thought → Action → Observation → Result)
- **Tools pré-LLM** : `vader_sentiment` + `detect_sarcasm_markers` (grounding déterministe)
- Classification du sentiment global (positive / neutral / negative)
- Production du `Score_Sentiment` [0-100]
- Pipeline anti-hallucination : `safe_llm_call` → `SentimentValidator` → retry ×3
- Comportement de fallback VADER quand le LLM est indisponible

> **Double mock requis** : `safe_llm_call` fait un import local `from models.llm_loader import get_llm`.
> Il faut patcher **à la fois** `agents.a3_sentiment.get_llm` (garde du nœud) ET `models.llm_loader.get_llm` (appelé par `safe_llm_call`).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from agents.a3_sentiment import a3_sentiment

## 1. Corpus de test

In [ ]:
CLEANED_COMMENTS = [
    {"text": "This is a great tutorial on machine learning!", "cleaned_text": "this is a great tutorial on machine learning!"},
    {"text": "Very informative, learned a lot about neural networks.", "cleaned_text": "very informative, learned a lot about neural networks."},
    {"text": "Excellent explanation of gradient descent.", "cleaned_text": "excellent explanation of gradient descent."},
    {"text": "Best course I have ever taken.", "cleaned_text": "best course i have ever taken."},
    {"text": "Could you cover backpropagation in more detail?", "cleaned_text": "could you cover backpropagation in more detail?"},
]

STATE = {"cleaned_comments": CLEANED_COMMENTS}
print(f"{len(CLEANED_COMMENTS)} commentaires chargés.")

## 2. A3 avec LLM mocké — sentiment positif

In [ ]:
# v3.0 — schéma JSON ReAct (reasoning + sarcasm_detected + confidence)
mock_response = MagicMock()
mock_response.content = json.dumps({
    "reasoning": (
        "Thought 1: VADER scores positive (compound=0.82). "
        "Thought 2: No sarcasm markers detected. "
        "Thought 3: Overall tone is genuinely enthusiastic."
    ),
    "sentiment_label":  "positive",
    "sentiment_score":  0.82,      # [0-1] — l'agent multiplie par 100
    "confidence":       0.91,
    "sarcasm_detected": False,
    "rationale": "The corpus expresses strong enthusiasm and positive feedback.",
})

mock_llm = MagicMock()
mock_llm.invoke.return_value = mock_response

# Double mock : garde du nœud + safe_llm_call (import local)
with patch("agents.a3_sentiment.get_llm", return_value=mock_llm), \
     patch("models.llm_loader.get_llm",   return_value=mock_llm):
    result = a3_sentiment(STATE)

s = result["sentiment"]
print(f"Label          : {s['sentiment_label']}")
print(f"Score          : {s['sentiment_score']} / 100  (0.82 × 100)")
print(f"Confidence     : {s.get('confidence')}")
print(f"VADER signal   : {s.get('vader_signal')}")
print(f"Sarcasm signal : {s.get('sarcasm_signal')}")
print(f"Rationale      : {s.get('rationale')}")
print(f"hallucination_flags : {result.get('hallucination_flags', [])}")
assert s["sentiment_label"] == "positive"
assert s["sentiment_score"] == 82.0
print("\nOK — score [0-1] → [0-100] confirmé.")

## 3. Fallback — LLM indisponible

In [ ]:
# v3.0 — fallback VADER (pas de LLM) : label dépend du corpus, score est dynamique
with patch("agents.a3_sentiment.get_llm", return_value=None):
    fallback = a3_sentiment(STATE)

s = fallback["sentiment"]
print(f"Fallback label : {s['sentiment_label']}")
print(f"Fallback score : {s['sentiment_score']}")

# v3.0 : le fallback utilise VADER — le label dépend du corpus
VALID_LABELS = {"positive", "neutral", "negative"}
assert s["sentiment_label"] in VALID_LABELS, f"Label inattendu : {s['sentiment_label']}"
assert 0.0 <= s["sentiment_score"] <= 100.0, f"Score hors [0-100] : {s['sentiment_score']}"
# Le corpus de test est positif → VADER → label="positive", score=70.0
assert s["sentiment_label"] == "positive", (
    f"Attendu 'positive' (corpus positif + VADER) — reçu : {s['sentiment_label']}"
)
assert s["sentiment_score"] == 70.0, f"Attendu 70.0 — reçu : {s['sentiment_score']}"
print("\nFallback v3.0 conforme (VADER-based, pas de LLM).")

## Résumé A3 — PRD v3.0

| Comportement | Résultat |
|---|---|
| **ReAct** (Thought 1→2→3 → Result) | Schéma JSON `reasoning` injecté dans le prompt |
| **Tool 1** : `vader_sentiment` | Score VADER injecté avant le LLM |
| **Tool 2** : `detect_sarcasm_markers` | Correction sarcasme/ironie |
| Score [0,1] → [0,100] | Multiplication par 100 confirmée |
| `sentiment_score=0.82` → `82.0` | Confirmé |
| `confidence`, `sarcasm_detected` | Champs v3.0 retournés par le LLM |
| `vader_signal`, `sarcasm_signal` | Enrichis post-LLM dans le résultat |
| `hallucination_flags` | Propagés dans le state LangGraph |
| LLM indisponible | Fallback **VADER** — label dynamique selon corpus |
| Double mock requis | `agents.a3_sentiment.get_llm` + `models.llm_loader.get_llm` |

> **Poids w1 = 0.35** — `Score_Global = 0.35×S + 0.40×D + 0.25×N`  
> **Modèle cible Kaggle** : Phi-3-mini-4k-instruct (float16, ~4 Go VRAM)